# Fitting experimental measurements with tensors

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/measurement_fitting)

A (synthetic) lab measurement of a damped oscillator, with an error bar $\sigma_i$ on every point, is fitted by maximum likelihood. The measurement model has an explicit error term $\varepsilon_i$ whose spread includes an unknown extra scatter $s$; autograd minimises $-2\ln L$, the Jacobian and Hessian give the parameter uncertainties, and a linear sub-problem shows least squares as a single lstsq solve.

**Vector form**

$$x_i = f(t_i;\boldsymbol\theta) + \varepsilon_i,\quad \varepsilon_i \sim N(0, \sigma_i^2 + s^2),\quad f = A\,e^{-\gamma t}\cos(\omega t + \varphi) + c,\qquad -2\ln L = \sum_i \left[\frac{(x_i - f_i)^2}{\sigma_i^2 + s^2} + \ln 2\pi(\sigma_i^2 + s^2)\right]$$

**Index (tensor) form**

$$r_i = \frac{x_i - f(t_i;\theta)}{\sigma_{tot,i}},\ \ \sigma_{tot,i}^2 = \sigma_i^2 + s^2 \qquad -2\ln L = r_i\,r_i + \textstyle\sum_i \ln 2\pi\sigma_{tot,i}^2 \qquad J_{ip} = \frac{\partial r_i}{\partial \theta_p} \qquad C_{pq} = (J_{ip}J_{iq})^{-1} \qquad f_i = D_{ik}\,\beta_k$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $t_i,\ x_i,\ \sigma_i$ | `load_measurements(dev)` | (N,) each | time, measured position and quoted error bar of point $i$, read from measurements.csv |
| $\theta_p,\ \ln s$ | `p` | (6,) | $(A, \gamma, \omega, \varphi, c, \ln s)$; fitting $\ln s$ keeps the jitter $s$ positive |
| $f(t_i;\theta)$ | `model(t, p)` | (N,) | the damped-oscillator model at every measured time |
| $\sigma_{tot,i}$ | `total_sigma(sigma, p[5])` | (N,) | spread of the error term $\varepsilon_i$: error bar and extra scatter added in quadrature |
| $r_i\,r_i + \sum_i \ln 2\pi\sigma_{tot,i}^2$ | `neg2_log_likelihood(p, t, x, sigma)` | scalar | repeated $i$ ⇒ summed; with $s = 0$ it is $\chi^2$ plus a constant |
| $J_{ip}$ | `autograd.functional.jacobian(residuals, p[:5])` | (N, 5) | how each normalised residual changes with each parameter |
| $(J_{ip}J_{iq})^{-1}$ | `inv(einsum('ip,iq->pq', J, J))` | (5, 5) | Gauss–Newton covariance $C$: uncertainties $\sqrt{C_{pp}}$ ($i$ summed) |
| $2\,H^{-1}$ | `covariance_from_hessian(p, t, x, sigma)` | (6, 6) | covariance including $\ln s$, from the Hessian of $-2\ln L$ |
| $D_{ik}\,\beta_k$ | `lstsq(D / sigma[:, None], x / sigma)` | (N, 3) → (3,) | for fixed $\gamma, \omega$ the model is linear in $\beta = (A\cos\varphi, A\sin\varphi, c)$ |

A measurement is never exactly the model: $x_i = f(t_i;\theta) + \varepsilon_i$, and the error term $\varepsilon_i$ is what the statistics are about. If the quoted error bars were the whole story, $\varepsilon_i \sim N(0, \sigma_i^2)$ and maximising the likelihood would be the same as minimising $\chi^2 = \sum_i ((x_i - f_i)/\sigma_i)^2$ (weighted least squares). Real experiments often scatter more than their error bars say (vibrations, an unmodelled drift), so here the spread is $\sigma_{tot,i}^2 = \sigma_i^2 + s^2$ with an unknown jitter $s$ fitted along with the five physical parameters. The term $\ln 2\pi\sigma_{tot,i}^2$ in $-2\ln L$ stops the fit from explaining everything away with a huge $s$. The data in measurements.csv are synthetic, generated with $s = 0.8$ mm on top of error bars of 0.5–1 mm: a plain $\chi^2$ fit ends with $\chi^2/\mathrm{dof} \approx 2.5$, the sign of underestimated errors, while with $s$ fitted the normalised residuals $r_i = (x_i - f_i)/\sigma_{tot,i}$ follow $N(0,1)$ and $\chi^2/\mathrm{dof} \approx 1$. Autograd gives the gradient of $-2\ln L$ for Adam (the animated iterations) and L-BFGS (the final polish). The uncertainties come from the curvature of $-2\ln L$ at the minimum: $C = (J^TJ)^{-1}$ with the Jacobian $J_{ip} = \partial r_i/\partial\theta_p$, or $C = 2H^{-1}$ with the full Hessian when $s$ is included. Linear vs nonlinear least squares: for fixed $\gamma$ and $\omega$ the model is linear in $(A\cos\varphi, A\sin\varphi, c)$, so the best values follow from one lstsq solve (no iterations, a unique answer), which is how the starting guess is made from $\omega_0, \gamma_0$; $\gamma$ and $\omega$ enter nonlinearly, so they need an iterative search that can get stuck in a local minimum. Try a poor $\omega_0$ (the fit may lock on a wrong frequency), a large learning rate, or switch the jitter off. Simplifications: independent Gaussian errors, no error on $t$, and covariances from the local curvature only.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/measurement_fitting/main.py
"""Main loop: load data → initial guess → fit (maximum likelihood) → uncertainties → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    t, x, sigma = tensors.load_measurements(dev)  # (N,) each
    N = len(t)
    jitter = params["jitter"] == "on"
    p0 = tensors.initial_parameters(t, x, sigma, params["omega0"], params["gamma0"])  # (6,)

    p, history, _ = solve.fit(p0, t, x, sigma, params["lr"], params["iterations"], jitter)
    chi2_hist = torch.stack([torch.sum(equation.residuals(q, t, x, sigma, jitter) ** 2) for q in history])

    # uncertainties: Hessian of -2 ln L (with s), or Gauss–Newton (JᵀJ)⁻¹ for plain χ²
    if jitter:
        C = equation.covariance_from_hessian(p, t, x, sigma)  # (6, 6)
    else:
        C = equation.covariance_from_jacobian(equation.jacobian(p, t, x, sigma, jitter=False))  # (5, 5)
    err = C.diagonal().sqrt()
    print(f"N = {N} points, {'maximum likelihood with jitter s' if jitter else 'weighted least squares (s = 0)'}")
    for i, name in enumerate(equation.NAMES[: len(err)]):
        print(f"  {name:7s} = {p[i].item():9.4f} ± {err[i].item():.4f}")
    if jitter:
        s = p[5].exp().item()
        print(f"  s       = {s:9.4f} ± {s * err[5].item():.4f} mm   (extra scatter beyond the error bars)")

    # χ²/dof with and without the error term s
    r = equation.residuals(p, t, x, sigma, jitter)
    print(f"χ²/dof using σ_tot = {(r @ r).item() / (N - len(err)):.3f}   (should be ≈ 1)")
    p_ls, _, _ = solve.fit(p0, t, x, sigma, params["lr"], params["iterations"], jitter=False)
    print(
        f"χ²/dof using σ_i only (weighted least squares fit) = {equation.chi2(p_ls, t, x, sigma).item() / (N - 5):.3f}"
    )

    # the linear sub-problem at the fitted γ, ω (and s): lstsq and normal equations give the same
    # β = (A cos φ, A sin φ, c), and the same A, φ, c as the nonlinear fit
    w = equation.total_sigma(sigma, p[5]) if jitter else sigma
    beta = equation.linear_fit(t, x, w, p[1].item(), p[2].item())
    beta_ne = equation.normal_equations(t, x, w, p[1].item(), p[2].item())
    A, phi = equation.amplitude_phase(beta)
    print(f"linear sub-problem at fitted γ, ω: lstsq β = {[round(b, 4) for b in beta.tolist()]}")
    print(
        f"  normal equations β = {[round(b, 4) for b in beta_ne.tolist()]}  →  A = {A.item():.4f}, φ = {phi.item():.4f}"
    )
    return plot.animate(t, x, sigma, history, chi2_hist, jitter, out_dir / "measurement_fitting.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/measurement_fitting/equation.py
r"""Physics → tensor translation: a measurement model with an explicit error term, and its likelihood.

Each measured point is the model plus an error:

    x_i = f(t_i; θ) + ε_i,      ε_i ~ N(0, σ_{tot,i}²),      σ_{tot,i}² = σ_i² + s²

    f(t; θ) = A e^{-γt} cos(ωt + φ) + c,       θ = (A, γ, ω, φ, c)

σ_i is the error bar quoted with point i; s is an extra scatter ("jitter") that the error bars missed,
fitted together with θ. All parameters live in one vector p = (A, γ, ω, φ, c, ln s), shape (6,);
s = e^{ln s} stays positive whatever the optimiser does.

Maximum likelihood: minimise

    -2 ln L = Σ_i [ r_i² + ln(2π σ_{tot,i}²) ],     r_i = (x_i - f_i) / σ_{tot,i}     (i summed)

With s = 0 the log term is a constant and -2 ln L = χ² + const, χ² = Σ_i ((x_i - f_i)/σ_i)²:
weighted least squares is maximum likelihood when the error bars are right.

Uncertainties: J_{ip} = ∂r_i/∂θ_p, and C_{pq} = (J_{ip} J_{iq})⁻¹  (i summed: Gauss–Newton), or, including s,
C = 2 H⁻¹ with H_{pq} = ∂²(-2 ln L)/∂p_p ∂p_q (the Hessian).
"""

import math

import torch

NAMES = ["A", "gamma", "omega", "phi", "offset", "log_s"]


def model(t: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """f_i = A e^{-γ t_i} cos(ω t_i + φ) + c.   t: (N,), theta: (≥5,) → (N,)."""
    A, gamma, omega, phi, offset = theta[0], theta[1], theta[2], theta[3], theta[4]
    return A * torch.exp(-gamma * t) * torch.cos(omega * t + phi) + offset


def total_sigma(sigma: torch.Tensor, log_s: torch.Tensor) -> torch.Tensor:
    """σ_{tot,i} = √(σ_i² + s²): quoted error bar and extra scatter add in quadrature.   (N,)"""
    return torch.sqrt(sigma**2 + torch.exp(2 * log_s))


def residuals(p: torch.Tensor, t, x, sigma, jitter: bool = True) -> torch.Tensor:
    """Normalised residuals r_i = (x_i - f_i) / σ_{tot,i} (σ_i alone when jitter is off).   (N,)"""
    s_tot = total_sigma(sigma, p[5]) if jitter else sigma
    return (x - model(t, p)) / s_tot


def neg2_log_likelihood(p: torch.Tensor, t, x, sigma) -> torch.Tensor:
    """-2 ln L = Σ_i [ r_i² + ln(2π σ_{tot,i}²) ]: the misfit plus a penalty for blaming it on a large s."""
    s_tot = total_sigma(sigma, p[5])
    r = (x - model(t, p)) / s_tot
    return torch.sum(r**2 + torch.log(2 * math.pi * s_tot**2))


def chi2(p: torch.Tensor, t, x, sigma) -> torch.Tensor:
    """χ² = Σ_i ((x_i - f_i)/σ_i)²: weighted least squares, i.e. -2 ln L with s = 0 (minus a constant)."""
    r = residuals(p, t, x, sigma, jitter=False)
    return torch.einsum("i,i->", r, r)


def jacobian(p: torch.Tensor, t, x, sigma, jitter: bool = True) -> torch.Tensor:
    """J_{ip} = ∂r_i/∂θ_p for the five model parameters (s held fixed), by autograd.   (N, 5)"""
    rest = p[5:]
    return torch.autograd.functional.jacobian(
        lambda theta: residuals(torch.cat([theta, rest]), t, x, sigma, jitter), p[:5]
    )


def covariance_from_jacobian(J: torch.Tensor) -> torch.Tensor:
    """C_{pq} = (J_{ip} J_{iq})⁻¹, i summed: the Gauss–Newton covariance of θ.   (N, 5) → (5, 5)"""
    return torch.linalg.inv(torch.einsum("ip,iq->pq", J, J))


def covariance_from_hessian(p: torch.Tensor, t, x, sigma) -> torch.Tensor:
    """C = 2 H⁻¹, H = Hessian of -2 ln L in all six parameters (including ln s).   (6, 6)"""
    H = torch.autograd.functional.hessian(lambda q: neg2_log_likelihood(q, t, x, sigma), p)
    return 2 * torch.linalg.inv(H)


# --- the linear sub-problem: for fixed γ, ω the model is linear in (a, b, c) ---------------------------


def design_matrix(t: torch.Tensor, gamma: float, omega: float) -> torch.Tensor:
    """D_{ik} with columns e^{-γt} cos ωt, -e^{-γt} sin ωt, 1, so f_i = D_{ik} β_k with β = (A cos φ, A sin φ, c).

    (cos(ωt + φ) = cos ωt cos φ - sin ωt sin φ.)   t: (N,) → (N, 3)
    """
    e = torch.exp(-gamma * t)
    return torch.stack([e * torch.cos(omega * t), -e * torch.sin(omega * t), torch.ones_like(t)], dim=1)


def linear_fit(t, x, sigma, gamma: float, omega: float) -> torch.Tensor:
    """Weighted linear least squares by torch.linalg.lstsq: rows divided by σ_i.   → β (3,)"""
    D = design_matrix(t, gamma, omega) / sigma[:, None]  # (N, 3)
    return torch.linalg.lstsq(D, (x / sigma)[:, None]).solution[:, 0]


def normal_equations(t, x, sigma, gamma: float, omega: float) -> torch.Tensor:
    """The same β from the normal equations (D_{ik} w_i D_{il}) β_l = D_{ik} w_i x_i, w_i = 1/σ_i².   (3,)"""
    D = design_matrix(t, gamma, omega)
    w = 1 / sigma**2
    M = torch.einsum("ik,i,il->kl", D, w, D)  # (3, 3)
    rhs = torch.einsum("ik,i,i->k", D, w, x)  # (3,)
    return torch.linalg.solve(M, rhs)


def amplitude_phase(beta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """(a, b) = (A cos φ, A sin φ) → (A, φ)."""
    return torch.hypot(beta[0], beta[1]), torch.atan2(beta[1], beta[0])


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/measurement_fitting/tensors.py
"""Build the concrete tensors: load the measurements and make a starting guess for the parameters.

measurements.csv (next to this file) holds a synthetic lab run: time t_s, position x_mm, error bar sigma_mm.
float64 is used throughout: the Hessian and (JᵀJ)⁻¹ are more trustworthy in double precision.
"""

import math
from pathlib import Path

import torch

from . import equation

DATA = Path(__file__).parent / "measurements.csv"
DTYPE = torch.float64


def load_measurements(device: torch.device, path: Path = DATA) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Read the CSV with plain Python (lines starting with # are comments): t, x, σ, each of shape (N,)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            if line.startswith("t_s"):  # the header row: t_s,x_mm,sigma_mm
                continue
            rows.append([float(v) for v in line.split(",")])
    data = torch.tensor(rows, dtype=DTYPE, device=device)  # (N, 3)
    return data[:, 0], data[:, 1], data[:, 2]


def initial_parameters(t, x, sigma, omega0: float, gamma0: float) -> torch.Tensor:
    """p0 = (A, γ, ω, φ, c, ln s), shape (6,).

    Only γ0 and ω0 are guessed; for those values the model is linear in (A cos φ, A sin φ, c), so we get
    A, φ and c from one linear least-squares solve. The jitter starts small, s = 0.1 mm.
    """
    beta = equation.linear_fit(t, x, sigma, gamma0, omega0)
    A, phi = equation.amplitude_phase(beta)
    return torch.tensor(
        [A.item(), gamma0, omega0, phi.item(), beta[2].item(), math.log(0.1)], dtype=DTYPE, device=t.device
    )


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/measurement_fitting/solve.py
"""Minimise -2 ln L (or χ²) with autograd: Adam for the animated iterations, then L-BFGS to polish."""

import torch

from problems._common.progress import steps

from .equation import chi2, neg2_log_likelihood


def objective(p: torch.Tensor, t, x, sigma, jitter: bool) -> torch.Tensor:
    """-2 ln L with the jitter s fitted, or plain χ² (s = 0) when jitter is off."""
    return neg2_log_likelihood(p, t, x, sigma) if jitter else chi2(p, t, x, sigma)


def fit(p0: torch.Tensor, t, x, sigma, lr: float, iterations: int, jitter: bool = True):
    """Return (p_best (6,), history (K, 6), loss (K,)): parameters after every Adam step, the last row after L-BFGS.

    With jitter off, ln s gets no gradient and stays where it started; it is simply unused.
    """
    p = p0.clone().requires_grad_(True)
    history, losses = [p.detach().clone()], [objective(p, t, x, sigma, jitter).item()]

    adam = torch.optim.Adam([p], lr=lr)
    for _ in steps(iterations, "Adam iterations"):
        adam.zero_grad()
        loss = objective(p, t, x, sigma, jitter)
        loss.backward()  # autograd: ∂loss/∂p_p for all six parameters at once
        adam.step()
        history.append(p.detach().clone())
        losses.append(objective(p, t, x, sigma, jitter).item())

    lbfgs = torch.optim.LBFGS(
        [p], max_iter=200, tolerance_grad=1e-10, tolerance_change=1e-14, line_search_fn="strong_wolfe"
    )

    def closure():
        lbfgs.zero_grad()
        loss = objective(p, t, x, sigma, jitter)
        loss.backward()
        return loss

    lbfgs.step(closure)
    history.append(p.detach().clone())
    losses.append(objective(p, t, x, sigma, jitter).item())
    return p.detach(), torch.stack(history), torch.tensor(losses)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/measurement_fitting/plot.py
"""Animate the fit: data with error bars, the model and its ±σ_tot error band, normalised residuals, χ² vs iteration."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import model, total_sigma


def animate(t, x, sigma, history: torch.Tensor, chi2_hist: torch.Tensor, jitter: bool, path: Path) -> Path:
    tn, xn, sn = (a.cpu().numpy() for a in (t, x, sigma))
    hist = history.cpu()
    K = len(hist)
    chi = chi2_hist.cpu().numpy()
    fine = torch.linspace(0, tn[-1], 400, dtype=hist.dtype)
    sig_fine = torch.from_numpy(np.interp(fine.numpy(), tn, sn))  # error bars interpolated onto the fine grid
    curves = np.stack([model(fine, p).numpy() for p in hist])  # (K, 400)
    at_data = np.stack([model(t.cpu(), p).numpy() for p in hist])  # (K, N)
    if jitter:
        s_fine = np.stack([total_sigma(sig_fine, p[5]).numpy() for p in hist])
        s_data = np.stack([total_sigma(sigma.cpu(), p[5]).numpy() for p in hist])
    else:
        s_fine = np.tile(sig_fine.numpy(), (K, 1))
        s_data = np.tile(sn, (K, 1))
    resid = (xn - at_data) / s_data  # (K, N) normalised residuals r_i
    fn = fine.numpy()
    dof = len(tn) - (6 if jitter else 5)

    fig = plt.figure(figsize=(8.2, 4.6))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], height_ratios=[1.6, 1])
    ax_d = fig.add_subplot(grid[0, 0])
    ax_r = fig.add_subplot(grid[1, 0], sharex=ax_d)
    ax_c = fig.add_subplot(grid[0, 1])
    ax_h = fig.add_subplot(grid[1, 1])

    # --- data with error bars, model curve and its ±σ_tot band ---
    ax_d.errorbar(tn, xn, yerr=sn, fmt="o", ms=2.5, color="#111827", ecolor="#6b7280", elinewidth=0.8, capsize=0)
    (curve,) = ax_d.plot([], [], color="#dc2626", lw=1.6)
    band = [ax_d.fill_between(fn, curves[0] - s_fine[0], curves[0] + s_fine[0], color="#f87171", alpha=0.35, lw=0)]
    ax_d.set_ylim(xn.min() - 3, xn.max() + 3)
    ax_d.set_ylabel("x (mm)")
    ax_d.set_title(r"data $\pm\sigma_i$, model $f(t;\theta)$ $\pm\sigma_{tot}$", fontsize=10)
    ax_d.tick_params(labelbottom=False)
    label = ax_d.text(
        0.98,
        0.95,
        "",
        transform=ax_d.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="#e5e7eb", alpha=0.9),
    )

    # --- normalised residuals r_i = (x_i - f_i)/σ_tot,i ---
    ax_r.axhspan(-1, 1, color="#e5e7eb", lw=0)
    ax_r.axhline(0, color="#9ca3af", lw=1)
    (res_pts,) = ax_r.plot([], [], "o", ms=2.5, color="#2563eb")
    ax_r.set_ylim(-4, 4)
    ax_r.set_xlim(0, tn[-1])
    ax_r.set_xlabel("t (s)")
    ax_r.set_ylabel(r"$r_i$")

    # --- χ² vs iteration ---
    it = np.arange(K)
    ax_c.set_yscale("log")
    ax_c.set_xlim(0, K - 1)
    ax_c.set_ylim(min(chi.min(), dof) * 0.7, chi.max() * 1.5)
    ax_c.axhline(dof, color="#16a34a", ls="--", lw=1)
    ax_c.text(0.97 * (K - 1), dof * 1.12, f"dof = {dof}", fontsize=7.5, color="#16a34a", ha="right")
    (chi_line,) = ax_c.plot([], [], color="#2563eb", lw=1.4)
    ax_c.set_xlabel("iteration", labelpad=1)
    ax_c.set_title(r"$\chi^2=\sum_i r_i^2$ (last: L-BFGS)", fontsize=10)

    # --- histogram of normalised residuals vs N(0, 1) ---
    edges = np.linspace(-4, 4, 13)
    width = edges[1] - edges[0]
    centers = 0.5 * (edges[1:] + edges[:-1])
    bars = ax_h.bar(centers, np.zeros_like(centers), width=width * 0.9, color="#93c5fd", edgecolor="#2563eb", lw=0.5)
    g = np.linspace(-4, 4, 200)
    ax_h.plot(g, np.exp(-(g**2) / 2) / np.sqrt(2 * np.pi), color="#dc2626", lw=1.4)
    ax_h.set_xlim(-4, 4)
    ax_h.set_ylim(0, 0.75)
    ax_h.set_xlabel(r"$r_i$ vs $N(0,1)$", labelpad=1)

    def update(k: int):
        curve.set_data(fn, curves[k])
        band[0].remove()
        band[0] = ax_d.fill_between(fn, curves[k] - s_fine[k], curves[k] + s_fine[k], color="#f87171", alpha=0.35, lw=0)
        p = hist[k].numpy()
        stage = "L-BFGS" if k == K - 1 else f"Adam {k}"
        s_txt = f"  s={np.exp(p[5]):.2f}" if jitter else ""
        label.set_text(f"{stage}: A={p[0]:.1f} γ={p[1]:.3f} ω={p[2]:.3f}{s_txt}\nχ²/dof = {chi[k] / dof:.2f}")
        res_pts.set_data(tn, resid[k])
        chi_line.set_data(it[: k + 1], chi[: k + 1])
        counts = np.histogram(np.clip(resid[k], -3.99, 3.99), bins=edges)[0] / (len(tn) * width)
        for bar, h in zip(bars, counts):
            bar.set_height(h)

    fig.tight_layout()
    frames = frame_indices(K - 1, 63) + [K - 1]  # Adam steps, then the polished L-BFGS result
    frames += [K - 1] * 10  # hold the final fit for half a second
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'omega0': 4.6,
    'gamma0': 0.5,
    'lr': 0.03,
    'iterations': 300,
    'jitter': 'on'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.measurement_fitting")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.measurement_fitting.main")
problem_dir = Path("problems/others/measurement_fitting")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
